In [1]:
%load_ext autoreload
%autoreload 2

from meles.recognizer.arcface import ArcFaceRecognizer
from meles.aggregation.aggregator import NoAggregator, Aggregator
from meles.aggregation.clustering import MeanClusteringAggregator, MedianClusteringAggregator
from meles.aggregation.pooling import MeanPoolingAggregator, MaxPoolingAggregator, MinPoolingAggregator, \
    MedianPoolingAggregator
from meles.classifier import Classifier
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Allow the usage of progress bars on pandas dataframes
tqdm.pandas()

In [2]:
# Load the dataset from the experiment structure file.
DATA_PATH = Path("../../data")

experiment = pd.read_json(DATA_PATH.joinpath("ytfaces_experiment.json"))
experiment

,identity,video,frame,frame_suffix,path
0,Aaron_Eckhart,0,aligned_detect_0.555.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.555.jpg
1,Aaron_Eckhart,0,aligned_detect_0.556.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.556.jpg
2,Aaron_Eckhart,0,aligned_detect_0.557.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.557.jpg
3,Aaron_Eckhart,0,aligned_detect_0.558.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.558.jpg
4,Aaron_Eckhart,0,aligned_detect_0.559.jpg,.jpg,ytfaces/Aaron_Eckhart/0/aligned_detect_0.559.jpg
...,...,...,...,...,...
130722,Zulfiqar_Ahmed,4,aligned_detect_4.225.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.225.jpg
130723,Zulfiqar_Ahmed,4,aligned_detect_4.226.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.226.jpg
130724,Zulfiqar_Ahmed,4,aligned_detect_4.227.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.227.jpg
130725,Zulfiqar_Ahmed,4,aligned_detect_4.228.jpg,.jpg,ytfaces/Zulfiqar_Ahmed/4/aligned_detect_4.228.jpg


In [3]:
# Keep the first three videos of every identity in the train set and move the rest into the test set.
TRAIN_VIDEOS_PER_IDENTITY = 3

# Rank the distinct videos of each identity by their natural (ascending) order, so
# that the three lowest-numbered videos of every identity receive ranks 1, 2 and 3.
video_rank = experiment.groupby("identity")["video"].transform(
    lambda videos: videos.rank(method="dense")
)
is_train = video_rank <= TRAIN_VIDEOS_PER_IDENTITY

train = experiment[is_train].reset_index(drop=True)
test = experiment[~is_train].reset_index(drop=True)

# Every identity keeps at least its first video in the train set, so all test identities
# are guaranteed to be present in the train set as well.
assert set(test["identity"]).issubset(set(train["identity"]))

print(f"Train: {len(train):>8,} frames | {train.groupby(['identity', 'video']).ngroups:>5,} videos | {train['identity'].nunique():>4,} identities")
print(f"Test:  {len(test):>8,} frames | {test.groupby(['identity', 'video']).ngroups:>5,} videos | {test['identity'].nunique():>4,} identities")

Train:  110,367 frames | 1,599 videos |  533 identities
Test:    20,360 frames |   293 videos |  226 identities


In [4]:
# For every identity, embed all frames into a list of embeddings (with potentially different size).
def embed(agg: Aggregator, data: pd.DataFrame) -> pd.DataFrame:
    embeddings = (
        data.groupby('identity')['path']
        .progress_apply(lambda paths: agg.embed([str(DATA_PATH / path) for path in paths]))
        .explode()
        .rename('embedding')
        .reset_index()
        .dropna(subset=['embedding'])
    )
    return embeddings

In [10]:
# Initialize and configure the recognizer, aggregator and classifier. The classifier has to use the metric of the recognizer.
aggregator = MedianClusteringAggregator(ArcFaceRecognizer(), distance_threshold=None, min_size=3, n_clusters=3)
classifier = Classifier(metric=aggregator.metric())

In [11]:
train_embeddings = embed(aggregator, train)
train_embeddings.to_json(DATA_PATH / f"ytfaces_embeddings_train_{aggregator.name()}.json")
train_embeddings

100%|██████████| 533/533 [00:08<00:00, 65.56it/s]


,identity,embedding
0,Aaron_Eckhart,"[0.027312811464071274, 0.10410445928573608, -0..."
1,Aaron_Eckhart,"[-0.1883350908756256, 0.3415515422821045, -0.1..."
2,Aaron_Eckhart,"[-0.03668152540922165, -0.012408498674631119, ..."
3,Abdel_Aziz_Al-Hakim,"[-0.10868372768163681, -0.140267014503479, -0...."
4,Abdel_Aziz_Al-Hakim,"[0.15279380977153778, -0.023086968809366226, -..."
...,...,...
1541,Zoran_Djindjic,"[-0.03176141530275345, 0.021937068551778793, -..."
1542,Zoran_Djindjic,"[-0.1678241342306137, 0.3145098090171814, -0.0..."
1543,Zulfiqar_Ahmed,"[-0.019282144960016012, 0.004798530135303736, ..."
1544,Zulfiqar_Ahmed,"[-0.0005952049978077412, 0.1782679259777069, -..."


In [12]:
test_embeddings = embed(aggregator, test)
test_embeddings.to_json(DATA_PATH / f"ytfaces_embeddings_test_{aggregator.name()}.json")
test_embeddings

100%|██████████| 226/226 [00:01<00:00, 200.48it/s]


,identity,embedding
0,Adrian_Nastase,"[-0.06810168921947479, 0.12018158659338951, -0..."
1,Adrian_Nastase,"[0.01215954264625907, 0.2574305534362793, -0.2..."
2,Agbani_Darego,"[-0.10191865637898445, 0.33131591975688934, 0...."
3,Agbani_Darego,"[0.012127083726227283, 0.2543952465057373, -0...."
4,Agnes_Bruckner,"[-0.08921178430318832, -0.06492335349321365, 0..."
...,...,...
592,William_Delahunt,"[-0.09046942740678787, 0.2672269642353058, -0...."
593,William_McDonough,"[-0.09705588221549988, -0.008612689562141895, ..."
594,William_McDonough,"[-0.061100125312805176, -0.060277923941612244,..."
595,Yuri_Fedotov,"[-0.05750199034810066, -0.10054086148738861, 0..."


## Fit and evaluate the classifier

Fit the classifier on the train embeddings using the identity as the label, then classify every
test frame against the fitted gallery. We report the top-1 **accuracy** and the **rank-1** metric.
Since the classifier assigns each probe the identity of its single closest gallery neighbour, the
top-1 accuracy equals the rank-1 identification rate; the additional ranks show how quickly the
correct identity is recovered when more neighbours are considered.

In [13]:
# Fit the classifier on the train embeddings, using the identity as the label.
def to_matrix(embeddings: pd.DataFrame) -> np.ndarray:
    """Stack the per-frame embedding lists into a (num_frames, embedding_dim) matrix."""
    return np.vstack(embeddings["embedding"].to_numpy())

X_train = to_matrix(train_embeddings)
y_train = train_embeddings["identity"].to_numpy()

classifier.fit(X_train, y_train)

In [14]:
# Classify every test frame against the fitted gallery and evaluate the classifier.
from sklearn.metrics import accuracy_score

X_test = to_matrix(test_embeddings)
y_test = test_embeddings["identity"].to_numpy()

# Retrieve the ranked gallery neighbours (closest first) for every test frame.
RANKS = (1, 5, 10)
neighbor_labels, _ = classifier.classify(X_test, n_neighbors=max(RANKS))

# Top-1 accuracy: the closest neighbour's identity is the predicted identity.
y_pred = neighbor_labels[:, 0]
accuracy = accuracy_score(y_test, y_pred)

# Rank-k identification rate: the true identity appears within the k closest neighbours.
def rank_k_rate(k: int) -> float:
    hits = (neighbor_labels[:, :k] == y_test[:, None]).any(axis=1)
    return float(hits.mean())

print(f"Test frames: {len(y_test):,}")
print(f"Accuracy:    {accuracy:.4f}")
for k in RANKS:
    print(f"Rank-{k:<2}      {rank_k_rate(k):.4f}")

Test frames: 597
Accuracy:    0.5930
Rank-1       0.5930
Rank-5       0.6549
Rank-10      0.6817


None
Test frames: 20,360
Accuracy:    0.6347
Rank-1       0.6347
Rank-5       0.6813
Rank-10      0.7024

Max
Test frames: 226
Accuracy:    0.3673
Rank-1       0.3673
Rank-5       0.4602
Rank-10      0.5088

Mean
MeanPoolingAggregator
Test frames: 226
Accuracy:    0.5973
Rank-1       0.5973
Rank-5       0.6460
Rank-10      0.6770

Min
Test frames: 226
Accuracy:    0.3407
Rank-1       0.3407
Rank-5       0.4115
Rank-10      0.4602

Median
Test frames: 226
Accuracy:    0.6416
Rank-1       0.6416
Rank-5       0.6903
Rank-10      0.7124

Mean(n_clusters=3, min_size=3) clustering
Test frames: 597
Accuracy:    0.5930
Rank-1       0.5930
Rank-5       0.6549
Rank-10      0.6817

Mean(1.0) clustering
Test frames: 229
Accuracy:    0.5983
Rank-1       0.5983
Rank-5       0.6288
Rank-10      0.6507

Median(1.0) clustering
Test frames: 229
Accuracy:    0.6332
Rank-1       0.6332
Rank-5       0.6812
Rank-10      0.7031

Median(0.8, min_size=3) clustering
Test frames: 317
Accuracy:    0.5426
Rank-1       0.5426
Rank-5       0.5836
Rank-10      0.6088

Median(0.8) clustering
Test frames: 335
Accuracy:    0.5134
Rank-1       0.5134
Rank-5       0.5522
Rank-10      0.5761

Median(0.5) clustering
Test frames: 431
Accuracy:    0.4988
Rank-1       0.4988
Rank-5       0.5684
Rank-10      0.5893